# Lab 19: Tree-Based Models — Random Forests
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 40 min Core + 55 min AI + 25 min Extension

---

**Learning Objectives:**
- See a single tree memorise its training rows, and a forest of such trees generalise
- Compare a forest with OLS and Ridge on the same split, and read its out-of-bag score
- Compare impurity (MDI) and permutation importance, and explain why they disagree
- Tune `max_depth` and `max_features` by cross-validation
- Write your own prompt for a feature-importance chart, and check the AI's numbers against the lab's

**Dataset:** California Housing (scikit-learn): 20,640 block groups, 8 features, and the median
house value in units of $100,000. You have used it since Lab 4.

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one. Lab 16's
pipeline with `StandardScaler` and Lab 15's bias-variance curve are used again here.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions.
The AI-Assisted Expansion is required: as in Lab 16, you write the prompt yourself. The Extension
is optional.

**Plan on about 1 hour 55 minutes** for the required work: setup, Parts 1-3, the AI-Assisted
Expansion, and the README and submission.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - The answers to the interpretation questions, in text cells: two after Part 1, and one each after Parts 2 and 3.
> - **The AI-Assisted Expansion:** your own prompt, from scratch, asking AI for a function that charts a model's feature importances and gives them back as a table; a revised prompt for round 2 (and round 3 if you need it); a short checking log; and two answers. Your prompts, the log and the answers are what is graded, not the AI's code.
>
> **You run and read:** everything else. Setup and Parts 1-3 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The check cells in the AI section: run them; you need not follow their code. The AI's function is pasted, not typed.
>
> **The Extension** (optional) has one guided cell, then a task with no starter code that you write in a new cell. The **README** is text only: the prompt at the end is for documentation, not code.

## Setup

The first line of the setup cell upgrades scikit-learn to 1.9 or later, because the way trees
are built changed in 1.9: with an older version, the tree and forest numbers in this notebook
differ in the third or fourth decimal. The cell prints your scikit-learn version. If it shows a version below 1.9, use **Runtime → Restart session** and run this cell again.

Every model in this lab is given `random_state=RANDOM_STATE`, so running a cell twice gives the
same numbers.

`fetch_california_housing()` downloads the data the first time (a few seconds) and returns it in
parts: `.data` holds the features, `.target` the house values and `.feature_names` the column
names. `pd.DataFrame(data.data, columns=...)` puts the features in a table with those names.
`train_test_split` keeps 20% of the rows (`test_size=0.2`) as test rows, as in Lab 12.

In [ ]:
# GUIDED — run as-is
# Step 1: imports, and California Housing split into training and test rows
!pip install "scikit-learn>=1.9" -q

import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'scikit-learn {sklearn.__version__} (this lab\'s numbers assume 1.9 or later)')
print(f'Training set: {X_train.shape[0]} observations, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} observations')
print(f'Features: {list(X.columns)}')

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

**Expected output:** 16512 training rows with 8 features, 4128 test rows, and the eight feature
names from `MedInc` (median income, in $10,000s) to `Longitude`.

## Part 1: Decision Tree vs. Random Forest vs. Ridge (GUIDED — 14 min)

We compare four models on the same rows. The question: does the forest's added complexity buy
better predictions than simpler alternatives?

Step 2 fits them, each with `.fit(X, y)`, predictors first:
- a **single tree**, `DecisionTreeRegressor`, with no depth limit: it keeps splitting until it
  cannot;
- **OLS**, `LinearRegression`, as in Lab 12;
- **Ridge**, in a pipeline behind `StandardScaler`, as in Lab 16: the penalty falls evenly across
  the features only when they share one scale;
- a **random forest** of 100 trees, `RandomForestRegressor`. Each tree is grown on a bootstrap
  sample of the training rows, so it never sees about a third of them: its *out-of-bag* rows.
  `oob_score=True` asks the forest to score every training row using only the trees that never
  saw it.

Then it scores each model on the training rows and on the test rows. RMSE, the root of the mean
squared error, is in the target's units, $100,000s. The loop collects one dictionary per model,
and `pd.DataFrame(rows).set_index("Model")` turns the list into a table with one row per model.

In [ ]:
# GUIDED — run as-is
# Step 2: four models on the same training rows, scored on both sets
tree = DecisionTreeRegressor(random_state=RANDOM_STATE)      # no depth limit
tree.fit(X_train, y_train)

ols = LinearRegression()
ols.fit(X_train, y_train)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, oob_score=True, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

rows = []
for name, model in [('Single Tree', tree), ('OLS', ols),
                    ('Ridge', ridge), ('Random Forest', rf)]:
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    rows.append({
        'Model': name,
        'Train RMSE': np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Train R²': r2_score(y_train, train_pred),
        'Test R²': r2_score(y_test, test_pred),
    })

results_df = pd.DataFrame(rows).set_index('Model')
print(results_df.round(4))

**Expected output:**

| Model | Train RMSE | Test RMSE | Train R² | Test R² |
|---|---|---|---|---|
| Single Tree | 0.0000 | 0.7093 | 1.0000 | 0.6161 |
| OLS | 0.7197 | 0.7456 | 0.6126 | 0.5758 |
| Ridge | 0.7197 | 0.7456 | 0.6126 | 0.5758 |
| Random Forest | 0.1882 | 0.5063 | 0.9735 | 0.8044 |

Step 2, continued, looks inside the tree and the forest. `.get_n_leaves()` counts the tree's
final boxes: each leaf is one rectangle of feature space with one prediction, the mean of its
training rows. `rf.max_features` is how many features each split may choose from; `1.0` means
all of them. `.oob_score_` is the out-of-bag R². `results_df.loc[row, column]` reads one cell of
the table. In the prints, `:,` puts commas in the thousands and `:+.6f` shows six decimals with
the sign.

In [ ]:
# GUIDED — run as-is
# Step 2, continued: inside the tree and the forest
print(f"single tree: {tree.get_n_leaves():,} leaves for {len(X_train):,} training rows")
print(f"features each split of the forest may choose from: max_features = {rf.max_features}")
print(f"forest out-of-bag R²: {rf.oob_score_:.4f}   "
      f"test R²: {results_df.loc['Random Forest', 'Test R²']:.4f}")

gap = results_df.loc['Ridge', 'Test R²'] - results_df.loc['OLS', 'Test R²']
print(f"Ridge test R² minus OLS test R²: {gap:+.6f}")

**Expected output:** 15,870 leaves for 16,512 training rows; `max_features = 1.0`; an
out-of-bag R² of 0.8069 beside a test R² of 0.8044; and Ridge minus OLS of +0.000028.

Three notes on those lines:
- `max_features = 1.0` is the default the lecture asked you to check: `RandomForestRegressor` lets
  every split choose from every feature unless you say otherwise, so Step 2's forest decorrelates
  its trees only through the bootstrap: it is bagging. Part 3 tries the alternatives.
- The out-of-bag R² never touched a test row, yet lands close to the test R²: each training row is
  scored only by the trees that never drew it, so it is an honest held-out score. It is not exactly
  the test R² because the rows differ, and each row is scored by only about a third of the trees.
- The penalty bought Ridge essentially nothing here. With 16,512 training rows and only 8
  features, OLS has almost no variance for a penalty to buy back; regularization pays when p is
  large relative to n, as in Lab 16 (p = 200, n = 105).

### Interpretation Questions

1. Which model has the largest gap between its train R² and its test R²? Use the leaf count to explain where that gap comes from.
2. The forest's test R² is 0.23 above Ridge's, with the same eight features. What does that say about how the features relate to house values?

*Your answers here:*

1.
2.

## Part 2: Feature Importance — MDI vs. Permutation (GUIDED — 14 min)

Random forests tell us which features matter most for prediction. But remember: **predictive
importance is not causal importance.** Step 3 measures it two ways, for Step 2's forest.

- **MDI** (mean decrease in impurity), `rf.feature_importances_`, comes free with training. Each
  split lowers the squared error inside its box; MDI adds up those drops for each feature, weighted
  by how many training rows reached the split, and scales them to sum to 1.
- **Permutation importance**, `permutation_importance(rf, X_test, y_test, n_repeats=10,
  random_state=RANDOM_STATE)`, uses the fitted forest and the test rows. It shuffles one column
  at a time, which breaks that feature's link with the outcome, and records how far the forest's
  test R² falls (R² is the score `permutation_importance` uses for a regressor unless told
  otherwise). It repeats each shuffle 10 times with different orders, seeded, and
  `.importances_mean` is the average fall.

`pd.Series(values, index=X.columns)` labels each number with its feature. The two Series become
the two columns of one table, sorted by permutation importance, largest first
(`ascending=False`).

In [ ]:
# GUIDED — run as-is
# Step 3: impurity (MDI) and permutation importance for Step 2's forest
mdi_importance = pd.Series(rf.feature_importances_, index=X.columns)

perm_result = permutation_importance(
    rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE
)
perm_importance = pd.Series(perm_result.importances_mean, index=X.columns)

importance_table = pd.DataFrame({'MDI': mdi_importance, 'permutation': perm_importance})
print(importance_table.sort_values('permutation', ascending=False).round(4))

**Expected output:**

| | MDI | permutation |
|---|---|---|
| MedInc | 0.5250 | 0.7307 |
| Latitude | 0.0889 | 0.4465 |
| Longitude | 0.0886 | 0.3376 |
| AveOccup | 0.1384 | 0.2026 |
| HouseAge | 0.0547 | 0.0726 |
| AveRooms | 0.0442 | 0.0272 |
| AveBedrms | 0.0297 | 0.0091 |
| Population | 0.0306 | 0.0080 |

The AI-Assisted Expansion asks you to reproduce the permutation column with a function written
by AI.

Step 3, continued, draws both columns side by side. A horizontal bar chart (`.plot.barh`) draws
the first row at the bottom, so each Series is sorted smallest first to put the longest bar on
top.

In [ ]:
# GUIDED — run as-is
# Step 3, continued: the two measures side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

mdi_importance.sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('MDI')
axes[0].set_xlabel('importance')

perm_importance.sort_values().plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Permutation')
axes[1].set_xlabel('importance')

plt.tight_layout()
plt.savefig('figures/ch19_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation Questions

1. Which features does MDI rate above permutation, and what do they have in common? Which does it rate below, and why?

*Your answer here:*

## Part 3: Hyperparameter Tuning (GUIDED — 12 min)

Step 4 tunes the forest by cross-validation, with `GridSearchCV`. It fits a fresh forest for every
combination in `param_grid`, on every fold, and keeps the combination with the best average score.
- `max_depth`: 10 levels, or `None`, no limit (as in Step 2).
- `max_features`: how many of the 8 features each split may choose from. `'sqrt'` is √8, rounded
  down to 2; `0.5` is half of them, 4. Step 2 used all 8.
- `cv=3`: three folds, as in Lab 15.
- scikit-learn always keeps the *largest* score, so to minimise the error it scores by minus the
  MSE, `'neg_mean_squared_error'`. `np.sqrt(-score)` turns it back into an RMSE.
- `n_jobs=-1` uses every processor core.

Count the work before you run it: 2 × 2 = 4 combinations × 3 folds = 12 forests, each on about
11,000 rows, plus one more to refit the winner on all the training rows: 13. That takes about a
minute on Colab. A 3 × 3 × 3 grid with 5 folds would be 136 forests, and you would still be
waiting when the lab ended.

After the search, `.best_params_` is the winning combination and `.best_score_` its score.
`.best_estimator_` is a forest with those settings, refitted on all the training rows.
`.cv_results_` holds every combination's settings (`'params'`) and average score
(`'mean_test_score'`), in the same order; `zip` walks through the two lists side by side.
`str()` turns `None` into text, and `:<5` left-aligns it in 5 characters, so the columns line up.

In [ ]:
# GUIDED — run as-is (about a minute: 13 forests)
# Step 4: tune max_depth and max_features by 3-fold cross-validation
param_grid = {
    'n_estimators': [100],
    'max_depth': [10, None],
    'max_features': ['sqrt', 0.5],
}
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

for params, score in zip(grid_search.cv_results_['params'],
                         grid_search.cv_results_['mean_test_score']):
    print(f"max_depth = {str(params['max_depth']):<5} max_features = "
          f"{str(params['max_features']):<5} CV RMSE {np.sqrt(-score):.4f}")

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}')

best_rf = grid_search.best_estimator_
best_pred = best_rf.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, best_pred))
test_r2 = r2_score(y_test, best_pred)
print(f'Tuned RF — Test RMSE: {test_rmse:.4f}, Test R²: {test_r2:.4f}')

**Expected output:** CV RMSEs of 0.5522 (depth 10, `'sqrt'`), 0.5382 (depth 10, 0.5), 0.5066
(no limit, `'sqrt'`) and 0.5048 (no limit, 0.5). Then `Best parameters: {'max_depth': None,
'max_features': 0.5, 'n_estimators': 100}`, `Best CV RMSE: 0.5048`, and
`Tuned RF — Test RMSE: 0.4969, Test R²: 0.8116`.

### Interpretation Questions

1. Within the grid, which setting mattered more? Relative to Step 2's forest, which setting did the tuned forest actually change?

*Your answer here:*

---

## AI-Assisted Expansion: a feature-importance chart, your prompt, checked by you

Step 3 took two cells to compute permutation importance and chart it. A function that does both
for any fitted model, and hands back the numbers as a table, is code you would reuse in every
project, and quick for AI to write. It also makes every choice your prompt leaves out, and most of
those choices change the numbers. As in Lab 16, no prompt is given: you write it.

**The target.** Given Step 2's forest and the test rows, your function must reproduce Step 3's
permutation column, all eight features, from **MedInc 0.7307** to **Population 0.0080**. Asked for
the top 3, it must give back MedInc, Latitude and Longitude, in that order, with the same numbers.

**What the check calls.** `importance_chart(model, X, y, k)`: an already fitted `model`, a
DataFrame `X`, the outcome `y`, and `k`, the number of features to keep. It draws a horizontal bar
chart of the top `k` and gives back a pandas DataFrame with columns `feature` and `importance`,
largest first. The check runs it on a copy of Step 2's `rf` with `X_test` and `y_test`, at `k = 8`
and `k = 3`, and counts the forests it fits while it runs (Step 3 fitted none).

**What a good prompt states.** Step 3 and the paragraph above hold every answer.
- the function's name, its arguments in order and their types, and what it gives back;
- the model: what it is, and that it arrives already fitted;
- which importance, and every setting that changes its numbers;
- how the top `k` are picked and sorted;
- the chart: its type, its labels, and which bar goes on top;
- what to leave out (example data, a model of its own), and a comment on each line.

**What to do**
1. Write your round 1 prompt in the cell below and send it, exactly as written, to Claude
   ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT, in a new chat.
2. Read the reply before you run anything. Paste only the function into the round 1 cell, and run
   the check below it.
3. If a line says `NO`, find the choice in the function that caused it, state that choice in a
   round 2 prompt, and paste and check the new function in the round 2 cells. A third round goes
   over round 2's function. Stop when every line says `yes` and you can explain each line, or when
   you decide to stop.
4. If everything says `yes` in round 1, use round 2 to state one choice your prompt never asked
   for, and check that it still matches.
5. Fill in the checking log and answer the two questions. Those and your prompts are graded, not
   the code.

**Reading the check.** For each `k` it prints Step 3's features in order beside the AI's, then one
line per feature: your number, the AI's, `same?`, and the % apart. Same forest, same rows, same
session: a correct function agrees in every printed digit, whether the rows arrive as a DataFrame
or an array and however many processor cores do the work. There is no allowed gap. The check does
not read the chart: compare its bars with the table yourself.

Keep the function's name and argument order. If the AI also wrote example data or fitted its own
model, leave that out: it can replace the lab's.

In [ ]:
# GUIDED — run as-is (you need not follow this code; the cells below use it)
# The check, as a function: the cells below run it once for each round.
# It saves the forest, the test rows and Step 3's numbers now, before anything is pasted.
import copy
from sklearn.ensemble._forest import BaseForest     # the class every scikit-learn forest builds on

saved_rf, saved_X, saved_y = rf, X_test.copy(), y_test.copy()
saved_pred = rf.predict(X_test)
saved_perm = perm_importance.sort_values(ascending=False)     # Step 3's column, largest first


def lab_data_unchanged():
    """True if rf, X_test and y_test still hold what the lab made."""
    # try runs the lines under it; if they raise an error (say rf is gone), except runs instead
    try:
        return (rf is saved_rf and X_test.equals(saved_X)
                and np.array_equal(y_test, saved_y)
                and np.array_equal(rf.predict(X_test), saved_pred))
    except Exception:
        return False


def show_pair(label, mine, ai):
    """One line: your number, the AI's, whether they agree, and how far apart they are."""
    # rtol=1e-9: equal to about 9 significant digits, so only rounding inside the computer passes
    same = "yes" if np.isclose(mine, ai, rtol=1e-9, atol=0) else "NO"
    pct = 100 * (ai - mine) / abs(mine)
    # a gap under 1%: sign and 2 significant digits, so a tiny one prints like +9.6e-06;
    # otherwise a whole number with its sign, like -200
    apart = f"{pct:+.2g}" if abs(pct) < 1 else f"{pct:+.0f}"
    print(f"  {label:<24}{mine:>10.4f}{ai:>10.4f}{same:>7}{apart:>11}%")


def describe(value):
    """What the function gave back, in a few words."""
    if isinstance(value, (pd.Series, np.ndarray, list, tuple)):
        shown = ", ".join(f"{v:.4f}" if isinstance(v, (int, float, np.number)) else type(v).__name__
                          for v in list(value)[:8])
        return f"a {type(value).__name__}: {shown}"
    return f"a {type(value).__name__}"


def as_table(result):
    """The DataFrame the function gave back, with lower-case column names, or None."""
    if isinstance(result, (tuple, list)):          # (table, chart) or (chart, table)
        tables = [r for r in result if isinstance(r, pd.DataFrame)]
        if tables:
            result = tables[0]
    if not isinstance(result, pd.DataFrame):
        return None
    table = result.rename(columns=lambda c: str(c).lower())
    if "feature" not in table.columns and str(table.index.name).lower() == "feature":
        table = table.reset_index().rename(columns=lambda c: str(c).lower())
    return table


def check_importance_chart(fn):
    """Run the AI's importance_chart on Step 2's forest and the test rows, beside Step 3."""
    if not lab_data_unchanged():
        print("The pasted code changed rf, X_test or y_test. "
              "Paste only the function, then rerun the notebook from the top.")
        return
    # callable(x) is True when x is a function that can be called
    if not callable(fn):
        print("importance_chart should be a function. Paste the function the AI wrote.")
        return
    real_fit = BaseForest.fit
    for k in (8, 3):
        fits = []

        def counting_fit(self, *args, **kwargs):
            fits.append(1)
            return real_fit(self, *args, **kwargs)

        BaseForest.fit = counting_fit        # every forest fitted during the call is counted
        try:
            # a copy of the forest and of the rows, so the function cannot change the lab's
            result = fn(copy.deepcopy(saved_rf), saved_X.copy(), saved_y.copy(), k)
        except Exception as err:
            print(f"With k = {k}, the AI's function stopped with an error. "
                  f"{type(err).__name__}: {err}")
            return
        finally:
            BaseForest.fit = real_fit
        table = as_table(result)
        if table is None or not {"feature", "importance"} <= set(table.columns):
            shown = describe(result) if table is None else f"a DataFrame with columns {list(table.columns)}"
            print(f"importance_chart gave back {shown}.")
            print("The check needs a DataFrame with a column named feature and one named importance.")
            return
        ai_names = [str(f) for f in table["feature"]]
        mine_names = list(saved_perm.index[:k])
        print(f"\nk = {k}")
        print(f"  Step 3's top {k}, in order : {', '.join(mine_names)}")
        print(f"  the AI's rows, in order : {', '.join(ai_names)}")
        print(f"  same features, same order? {'yes' if ai_names == mine_names else 'NO'}")
        print(f"  {'':<24}{'Step 3':>10}{'AI':>10}{'same?':>7}{'% apart':>12}")
        for name in mine_names:
            if name not in ai_names:
                print(f"  {name:<24}{saved_perm[name]:>10.4f}{'missing':>10}")
                continue
            try:
                ai_value = float(table["importance"][table["feature"].astype(str) == name].iloc[0])
            except Exception:
                print(f"  {name:<24}{saved_perm[name]:>10.4f}   not a number")
                continue
            show_pair(name, saved_perm[name], ai_value)
        print(f"  {'forests fitted by it':<24}{0:>10}{len(fits):>10}"
              f"{'yes' if not fits else 'NO':>7}")
    print("\nIf a line says NO, find out why before you trust the function.")

**Your prompt, round 1.** Write it here, then send it exactly as written.

*Your prompt here:*

In [ ]:
# Paste the importance_chart function from round 1 here, then run the check below.
# Paste only the function: if the AI also wrote example data or fitted a model, leave it out.

In [ ]:
# CHECK — round 1
# To rerun this check, rerun the round 1 paste cell first: it checks whatever was pasted last
# globals() holds every name defined so far in this notebook
if "importance_chart" not in globals():
    print("Paste the AI's importance_chart function in the cell above first, keeping that name, "
          "and run that cell.")
else:
    importance_chart_r1 = importance_chart      # kept, so the round 2 check can tell the two apart
    check_importance_chart(importance_chart_r1)

**Your prompt, round 2** (and round 3 under it, if you need one). Keep what worked in round 1,
and state what the AI had to choose on its own.

*Your prompt here:*

In [ ]:
# Paste the importance_chart function from round 2 here, then run the check below.
# Paste only the function: if the AI also wrote example data or fitted a model, leave it out.

In [ ]:
# CHECK — round 2 (and round 3)
# "is" asks whether two names hold the very same function: if so, nothing new was pasted
if "importance_chart" not in globals() or importance_chart is globals().get("importance_chart_r1"):
    print("Paste the AI's round 2 importance_chart function in the cell above first, "
          "keeping that name, and run that cell.")
else:
    check_importance_chart(importance_chart)

### Checking log

This is graded, with your prompts and the two answers below. One row per round (a third round,
if any, goes in round 2's row): what you checked, how you checked it, and what you found. A row
that names the line that differed, how far apart it was, and the line of the AI's code
responsible earns credit; "it worked" does not. The last row is the chart, which the check does
not read: compare its bars with the table of your final round.

| Round | What I checked | How | What I found |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| Chart |  |  |  |

### Questions

1. **Critique, in your own words.** Name one choice the AI made that your prompt never stated, quote the line where it makes it, and say whether you would keep it and why. (A choice that caused a `NO` counts. If everything said `yes`, name a line that would give wrong numbers or break on another model or data instead: for example a classifier, whose default score is accuracy, not R²; `X` as a NumPy array; a hard-coded seed.) Then quote the lines of your final function that compute the importances and the lines that sort them and keep the top `k`, and explain in 2-3 sentences, in your own words, what they do and why they give Step 3's numbers.
2. **Judgment: where you stopped.** Which round did you stop at, and why do you, or do you not, trust that function on a model and data the check has never seen? Then: when would you ask AI to write chart code like this, and when would you write it yourself?

*Your answers here:*

1.
2.

---

## Extension (Optional — 25 min): Partial Dependence Plots, and a Forest as a Classifier

**Partial dependence.** A partial dependence plot shows how the forest's prediction moves with one
feature, averaging over the values of all the others in the test rows. It shows the shape of a
relationship the forest found, curves and flat stretches included, which one OLS coefficient
cannot. Step 5 draws it for the tuned forest of Step 4, for `MedInc` and `AveOccup`
(`kind='average'`: the average curve only). Like importance, it describes the model, not a cause.

In [ ]:
# GUIDED — run as-is (optional)
# Step 5: partial dependence of the tuned forest on two features
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(12, 4))
PartialDependenceDisplay.from_estimator(
    best_rf, X_test, features=['MedInc', 'AveOccup'], kind='average', ax=ax
)
plt.suptitle('Partial dependence of the tuned forest')
plt.tight_layout()
plt.savefig('figures/ch19_partial_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

**A forest as a classifier.** There is no starter code: write it in the cell below.

Turn the house value into a yes/no outcome, 1 when `y` is above its median (`np.median(y)`) and 0
otherwise (`.astype(int)` turns True/False into 1/0). Split it as Step 1 did (`test_size=0.2`,
`random_state=RANDOM_STATE`). Fit `RandomForestClassifier(n_estimators=200,
random_state=RANDOM_STATE)` and, for comparison, Lab 17's logistic regression with the features
scaled first: `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`. Score each on the test rows by AUC, from Lab 18:
`roc_auc_score(y_true, scores)`, where the scores are `.predict_proba(X)[:, 1]`. Import the three
from `sklearn.ensemble`, `sklearn.linear_model` and `sklearn.metrics`. With these settings the
forest scores 0.9609 and the logistic regression 0.9082. Without the scaler, `LogisticRegression`
does not converge in 1,000 iterations on these features, which is why Lab 17 scaled them first.

**Your answers.** Does the forest beat logistic regression on AUC, and is the difference
practically significant? If you were building this for a real estate company, which would you
choose, and why? Consider accuracy, interpretability and the cost of keeping it running.

If you skip this task, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# EXTENSION (optional) — write your classification comparison here

*Your answers here:*

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Compared a single decision tree, OLS, Ridge and a random forest on
  California Housing: test R² [YOUR VALUE] for the tree, [YOUR VALUE]
  for Ridge and [YOUR VALUE] for the forest
* Compared impurity (MDI) and permutation importance: Latitude scored
  [YOUR VALUE] by MDI but [YOUR VALUE] by permutation
* Tuned the forest by cross-validation: test R² [YOUR VALUE]
* Wrote my own prompt asking an AI for a feature-importance chart, went
  [YOUR VALUE] rounds, and checked it against my numbers: [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Tree-Based Models: Random Forests
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab19-random-forests`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab19-random-forests`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab19-random-forests`, branch `main`,
   commit message `Lab 19 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.